# Person Segmentation Training (RGB / RGB+D)

Единый ноутбук для обучения U-Net моделей на COCO: базовой RGB-версии, early fusion с глубиной и late fusion.


In [ ]:
from pathlib import Path
import importlib
import sys

import torch

PROJECT_ROOT = Path.cwd().resolve().parent
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

# Оптимальное использование Tensor Cores на современных GPU
torch.set_float32_matmul_precision("medium")

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger

from config import load_config
from datamodules.coco_person import CocoPersonDataModule
from models import (
    UNetResNet50,
    UNetResNet50RGB,
    UNetResNet50RGBD,
    UNetResNet50LateFusion,
)
from person_seg_module import PersonSegModule


## Загружаем конфиг


In [ ]:
def _resolve_path(value, base_dir):
    if value in (None, ""):
        return None
    path = Path(value).expanduser()
    if not path.is_absolute():
        path = (base_dir / path).resolve()
    return path

CONFIG_NAME = "unet_rgbd.json"  # варианты: "unet_rgb.json", "unet_rgbd.json", "unet_late_fusion.json"
config_path = (PROJECT_ROOT / "configs" / CONFIG_NAME).resolve()
if not config_path.exists():
    raise FileNotFoundError(f"Config not found: {config_path}")

config = load_config(str(config_path))

data_cfg = config["data"].copy()
model_cfg = config["model"].copy()
training_cfg = config["training"].copy()
log_cfg = config.get("logging", {}).copy()

DATA_DIR = _resolve_path(data_cfg.get("data_dir"), config_path.parent)
TRAIN_ANN = _resolve_path(data_cfg.get("train_ann"), config_path.parent)
VAL_ANN = _resolve_path(data_cfg.get("val_ann"), config_path.parent)
TEST_ANN = _resolve_path(data_cfg.get("test_ann"), config_path.parent)
DEPTH_DIR = _resolve_path(data_cfg.get("depth_dir"), config_path.parent)
CHECKPOINT_DIR = _resolve_path(training_cfg.get("checkpoint_dir"), config_path.parent)
DEFAULT_ROOT_DIR = _resolve_path(log_cfg.get("default_root_dir"), config_path.parent)

print(f"Config: {config_path}")
print(f"Data dir: {DATA_DIR}")
print(f"Depth dir: {DEPTH_DIR}")


## Подготавливаем датамодуль


In [ ]:
seed = training_cfg.get("seed", 42)
pl.seed_everything(seed, workers=True)

if DATA_DIR is None or not DATA_DIR.exists():
    raise FileNotFoundError(f"COCO data dir not found: {DATA_DIR}")
if TRAIN_ANN is None or not TRAIN_ANN.exists():
    raise FileNotFoundError(f"Train annotations not found: {TRAIN_ANN}")
if VAL_ANN is None or not VAL_ANN.exists():
    raise FileNotFoundError(f"Val annotations not found: {VAL_ANN}")
if TEST_ANN is not None and not TEST_ANN.exists():
    raise FileNotFoundError(f"Test annotations not found: {TEST_ANN}")

use_depth = DEPTH_DIR is not None
if use_depth and not DEPTH_DIR.exists():
    raise FileNotFoundError(
        "Depth directory not found. Generate depth maps with tools/predict_depth_anydepth.py"
    )

module = CocoPersonDataModule(
    data_dir=str(DATA_DIR),
    train_ann=str(TRAIN_ANN),
    val_ann=str(VAL_ANN),
    test_ann=str(TEST_ANN) if TEST_ANN else None,
    image_size=tuple(data_cfg.get("image_size", [360, 480])),
    batch_size=data_cfg.get("batch_size", 4),
    num_workers=data_cfg.get("num_workers", 4),
    pin_memory=data_cfg.get("pin_memory", True),
    depth_dir=str(DEPTH_DIR) if use_depth else None,
)

module.setup("fit")
if TEST_ANN:
    module.setup("test")

print(f"Seed: {seed}")
print(f"Train samples: {len(module._train_dataset)}")
print(f"Val samples: {len(module._val_dataset)}")
if module._test_dataset:
    print(f"Test samples: {len(module._test_dataset)}")
print(f"Using depth: {use_depth}")


## Модель


In [ ]:
num_classes = 2
model_name = model_cfg.get("name")
model_target = model_cfg.get("target")

MODEL_REGISTRY = {
    "unet_rgb": UNetResNet50RGB,
    "unet_rgbd": UNetResNet50RGBD,
    "unet_late_fusion": UNetResNet50LateFusion,
    "unet": UNetResNet50,
}

model_cls = None
if model_target:
    module_path, class_name = model_target.rsplit(".", 1)
    model_module = importlib.import_module(module_path)
    model_cls = getattr(model_module, class_name)
elif model_name:
    model_cls = MODEL_REGISTRY.get(model_name.lower())

if model_cls is None:
    in_channels = model_cfg.get("in_channels", 3)
    if in_channels > 3 or (DEPTH_DIR is not None):
        model_cls = UNetResNet50RGBD
        model_cfg.setdefault("name", "unet_rgbd")
    else:
        model_cls = UNetResNet50RGB
        model_cfg.setdefault("name", "unet_rgb")
    print(f"[info] Falling back to {model_cls.__name__}")

model_kwargs = {k: v for k, v in model_cfg.items() if k not in {"name", "target"}}
model_kwargs.setdefault("pretrained", True)

model = model_cls(num_classes=num_classes, **model_kwargs)

lit_model = PersonSegModule(
    model=model,
    learning_rate=training_cfg.get("learning_rate", 1e-4),
    weight_decay=training_cfg.get("weight_decay", 1e-4),
    optimizer=training_cfg.get("optimizer", "adamw"),
)

print(f"Model: {model.__class__.__name__}")
print(f"Params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
print(f"Model config: {model_cfg}")


## Тренер и колбэки


In [ ]:
callbacks = []

if CHECKPOINT_DIR:
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    callbacks.append(
        ModelCheckpoint(
            dirpath=str(CHECKPOINT_DIR),
            filename=training_cfg.get("checkpoint_filename", "model-{epoch:02d}-{val_mIoU:.4f}"),
            save_top_k=training_cfg.get("checkpoint_top_k", 3),
            monitor=training_cfg.get("checkpoint_monitor", "val/mIoU"),
            mode=training_cfg.get("checkpoint_mode", "max"),
        )
    )

callbacks.append(LearningRateMonitor(logging_interval="step"))

logger = None
if log_cfg.get("enable_logger", True):
    if DEFAULT_ROOT_DIR:
        DEFAULT_ROOT_DIR.mkdir(parents=True, exist_ok=True)
    logger = TensorBoardLogger(
        save_dir=str(DEFAULT_ROOT_DIR or Path("logs")),
        name=log_cfg.get("name", "person_segmentation"),
        version=log_cfg.get("version", CONFIG_NAME.split(".")[0]),
    )

trainer = pl.Trainer(
    accelerator=training_cfg.get("accelerator", "auto"),
    devices=training_cfg.get("devices", "auto"),
    max_epochs=training_cfg.get("max_epochs", 40),
    precision=training_cfg.get("precision", "16-mixed"),
    gradient_clip_val=training_cfg.get("gradient_clip_val", 0.0),
    log_every_n_steps=training_cfg.get("log_every_n_steps", 20),
    accumulate_grad_batches=training_cfg.get("accumulate_grad_batches", 1),
    callbacks=callbacks,
    logger=logger,
    default_root_dir=str(DEFAULT_ROOT_DIR) if DEFAULT_ROOT_DIR else None,
)

print("Trainer configured")


## Обучение и оценка


In [ ]:
trainer.fit(lit_model, datamodule=module)


In [ ]:
if TEST_ANN:
    trainer.test(lit_model, datamodule=module)
